# MWE: electric charges of BSM particles not (or wrongly) assigned

`ModelBuilder.generateParticleModels` fills the electric charge of each BSM particle from the
decay vertices of the protomodel. Two defects:

1. **A vertex can be skipped**: the solver walks the vertices by index, but `ParticleModel.copy()`
   re-sorts them, so a vertex may never be visited. The particle model is then returned with a
   BSM charge `None`, without error.
2. **The sign of a BSM daughter is lost**: while its charge is unknown, a BSM particle is its own
   antiparticle, so `-1000024` is read as `1000024`.

Each case below builds a fake protomodel and prints the charges of the BSM particles in the
resulting particle models.

In [1]:
import warnings
warnings.simplefilter('ignore')

from uvbuilder.modelBuilder.modelBuilder import ModelBuilder
from uvbuilder.tools.reps import GlobalU1s, DIRAC_SPINOR


def charges(decays, masses, globalU1s=GlobalU1s()):
    '''BSM electric charges {pdg: charge} of every particle model built from the protomodel.'''
    builder = ModelBuilder(modelData={'masses': masses, 'decays': decays},
                           possible_dm_reps={'spin': [DIRAC_SPINOR], 'globalU1s': [globalU1s]})
    return [{p.pdg: p.eCharge for p in sorted(pm.bsm_particles, key=lambda p: p.pdg)}
            for pm in builder.generateParticleModels()]

## Case 1: skipped vertex

Z2 (1000023) -> C1^- W^+, with C1 (1000024) -> Z1 W^+.
**Expected:** Z1 = 0, C1 = +1, Z2 = 0.

In [2]:
charges(decays={1000022: {},
                1000024: {(1000022, 24): 1.0},
                1000023: {(-1000024, 24): 1.0}},
        masses={1000022: 100., 1000024: 200., 1000023: 300.})

[{1000022: 0, 1000023: None, 1000024: 1}]

## Case 2: sign of the BSM daughter lost

Same protomodel, with both charge states of the Z2 decay listed:
Z2 -> C1^- W^+ and Z2 -> C1^+ W^-.
**Expected:** Z1 = 0, C1 = +1, Z2 = 0.

In [3]:
charges(decays={1000022: {},
                1000024: {(1000022, 24): 1.0},
                1000023: {(-1000024, 24): 0.5, (1000024, -24): 0.5}},
        masses={1000022: 100., 1000024: 200., 1000023: 300.})

[{1000022: 0, 1000023: 2, 1000024: 1}]

## Case 3: realistic coloured protomodel (standard PDGs)

gluino (1000021) -> stop2 (2000006) tbar, stop2 -> Z1 t (baryon number conserved).
**Expected:** gluino = 0, stop2 = +2/3, Z1 = 0.

In [4]:
charges(decays={1000022: {},
                2000006: {(1000022, 6): 1.0},
                1000021: {(2000006, -6): 1.0}},
        masses={1000022: 100., 2000006: 800., 1000021: 1400.},
        globalU1s=GlobalU1s(B=0))

[{1000021: None, 1000022: 0, 2000006: 2/3}]